# 11 — PaliGemma Card Analysis via Ollama

Uses `jyan1/paligemma-mix-224` (Google PaliGemma, 224×224 input) running locally in Ollama  
to perform multi-task analysis on Pokémon / TCG card images.

## Tasks covered
| # | Task | PaliGemma prompt prefix |
|---|------|-------------------------|
| 1 | Card detection & bounding box | `detect pokemon card` |
| 2 | Card identification (name, set, rarity) | `caption en` / custom VQA |
| 3 | Condition assessment — corners, edges, surface | structured VQA |
| 4 | Centering estimation | `detect inner frame` / VQA |
| 5 | PSA grade prediction | chain-of-thought VQA |
| 6 | OCR (HP, card number, set symbol) | `ocr` |

**Prerequisites:** Ollama running at `localhost:11434` with `jyan1/paligemma-mix-224` pulled.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
OLLAMA_HOST  = "http://localhost:11434"
MODEL_NAME   = "jyan1/paligemma-mix-224"

# Default test images — change to any path you like
TEST_FRONT   = "cards/image0_front.jpeg"
TEST_BACK    = "cards/image0_back.jpeg"

# Show all intermediate viz panels
VERBOSE      = True

In [ ]:
# ── Imports & setup ───────────────────────────────────────────────────────────
import base64, json, re, textwrap, time
from pathlib import Path
from typing import Optional

import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from IPython.display import display, Markdown

# ── Verify Ollama is up ────────────────────────────────────────────────────────
try:
    tags = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5).json()
    names = [m["name"] for m in tags.get("models", [])]
    if MODEL_NAME not in names:
        print(f"⚠️  {MODEL_NAME} not found. Available: {names}")
    else:
        print(f"✅  Ollama up. Model '{MODEL_NAME}' ready.")
except Exception as e:
    print(f"❌  Cannot reach Ollama at {OLLAMA_HOST}: {e}")

## Helper — Ollama multimodal call

In [ ]:
def encode_image(path: str | Path) -> str:
    """Return base64-encoded JPEG for Ollama image field."""
    img = Image.open(path).convert("RGB")
    # PaliGemma expects 224×224; Ollama resizes internally but we can help
    # by keeping aspect-ratio letterbox resize at ~512 so details survive
    img.thumbnail((512, 512), Image.LANCZOS)
    from io import BytesIO
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=92)
    return base64.b64encode(buf.getvalue()).decode()


def paligemma(
    prompt: str,
    image_path: str | Path,
    *,
    temperature: float = 0.0,
    max_tokens: int = 512,
    timeout: int = 60,
) -> str:
    """
    Send a PaliGemma-style prompt + image to Ollama and return the text response.

    PaliGemma prompt conventions (from Google's PaliGemma paper):
      - 'detect <thing>'       → YOLO-style <loc0001> tokens (bounding boxes)
      - 'caption en'           → free-form English caption
      - 'ocr'                  → read all visible text
      - '<question>?'          → visual question answering
    """
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "images": [encode_image(image_path)],
        "options": {"temperature": temperature, "num_predict": max_tokens},
        "stream": False,
    }
    t0 = time.time()
    resp = requests.post(f"{OLLAMA_HOST}/api/generate", json=payload, timeout=timeout)
    elapsed = time.time() - t0
    resp.raise_for_status()
    text = resp.json().get("response", "").strip()
    if VERBOSE:
        print(f"  ⏱  {elapsed:.1f}s | prompt: '{prompt[:60]}...' " if len(prompt) > 60 else f"  ⏱  {elapsed:.1f}s | prompt: '{prompt}'")
    return text


def show_image(path: str | Path, title: str = "", ax=None):
    img = Image.open(path).convert("RGB")
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 7))
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    return img, ax

## Task 1 — Card Detection (`detect` task)

PaliGemma encodes bounding boxes as `<loc0NNN>` tokens in `y0 x0 y1 x1` order  
(each value is the coordinate divided by image dimension × 1024, zero-padded to 4 digits).

In [ ]:
def parse_paligemma_boxes(raw: str, img_w: int, img_h: int) -> list[dict]:
    """
    Parse PaliGemma detection output into pixel bounding boxes.

    Format: '<loc0123><loc0456><loc0789><loc0999> label ; ...'
    Each loc token is a 0-1023 value normalised by the image side.
    Sequence is: y0 x0 y1 x1 (top-left to bottom-right in normalised coords).
    """
    loc_pattern = re.compile(r'(<loc(\d{4})>){4}')
    full_pattern = re.compile(r'(<loc(\d{4})>)<loc(\d{4})><loc(\d{4})><loc(\d{4})>\s*([^;<\n]*)')
    # Match all 4-token groups
    token_re = re.compile(r'<loc(\d{4})>')
    tokens = token_re.findall(raw)

    boxes = []
    i = 0
    # Try to group every 4 tokens as one box; leftover tokens ignored
    # Extract labels between box groups
    label_re = re.compile(r'>\s*([a-zA-Z][^<;\n]{0,40}?)(?=\s*(?:<loc|;|$))')
    labels_raw = label_re.findall(raw)
    labels = [l.strip() for l in labels_raw] if labels_raw else []

    while i + 3 < len(tokens):
        y0, x0, y1, x1 = [int(t) / 1024 for t in tokens[i:i+4]]
        boxes.append({
            "x0": int(x0 * img_w), "y0": int(y0 * img_h),
            "x1": int(x1 * img_w), "y1": int(y1 * img_h),
            "label": labels[i // 4] if i // 4 < len(labels) else "?",
        })
        i += 4
    return boxes


def detect_card(image_path: str | Path) -> dict:
    """Run PaliGemma detect task and draw result."""
    img = Image.open(image_path).convert("RGB")
    w, h = img.size

    raw = paligemma("detect pokemon card", image_path)
    print(f"  Raw output: {raw!r}")

    boxes = parse_paligemma_boxes(raw, w, h)

    fig, ax = plt.subplots(figsize=(6, 8))
    ax.imshow(img)
    for b in boxes:
        rect = mpatches.Rectangle(
            (b["x0"], b["y0"]), b["x1"]-b["x0"], b["y1"]-b["y0"],
            linewidth=2, edgecolor="cyan", facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(b["x0"]+4, b["y0"]+14, b["label"], color="cyan",
                fontsize=9, fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.5, pad=2, edgecolor="none"))
    ax.set_title(f"detect pokemon card  ({len(boxes)} box{'es' if len(boxes)!=1 else ''})", fontsize=10)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    return {"raw": raw, "boxes": boxes}


det = detect_card(TEST_FRONT)

## Task 2 — Caption & Card Identification

In [ ]:
def identify_card(image_path: str | Path) -> dict:
    """Multi-prompt card identification: name, set, rarity, HP, artist."""
    prompts = {
        "caption":  "caption en",
        "name":     "What is the name of this Pokémon card?",
        "set":      "What Pokémon TCG set does this card belong to?",
        "rarity":   "What is the rarity of this card? (Common, Uncommon, Rare, Holo Rare, etc.)",
        "hp":       "What is the HP value printed on this card?",
        "card_num": "What is the card number and total printed on this card? (e.g. 4/102)",
    }
    results = {}
    print("Card Identification Results")
    print("═" * 50)
    for key, prompt in prompts.items():
        ans = paligemma(prompt, image_path, max_tokens=128)
        results[key] = ans
        label = key.replace("_", " ").title().ljust(12)
        print(f"  {label}: {ans}")
    return results


ident = identify_card(TEST_FRONT)

## Task 3 — Condition Assessment

Targeted prompts for each of the four PSA grading dimensions.

In [ ]:
CONDITION_PROMPTS = {
    "corners": (
        "Examine the four corners of this trading card closely. "
        "Describe any whitening, fraying, or rounding you see. "
        "Rate each corner: Sharp / Minor Wear / Moderate Wear / Heavy Wear."
    ),
    "edges": (
        "Examine all four edges of this trading card. "
        "Describe any chipping, roughness, or whitening along the edges. "
        "Rate each edge: Clean / Minor Chips / Moderate Chips / Heavy Damage."
    ),
    "surface": (
        "Examine the surface of this trading card. "
        "Describe any scratches, print lines, dents, creases, or loss of gloss. "
        "Rate surface quality: Pristine / Minor Issues / Moderate Issues / Heavy Damage."
    ),
    "centering": (
        "Examine the centering of this trading card. "
        "Describe how evenly the printed image is centered within the card borders. "
        "Estimate left/right border ratio and top/bottom border ratio (e.g. 55/45 L/R)."
    ),
    "overall": (
        "You are a PSA card grading expert. "
        "Based on the visible condition of this trading card, "
        "provide an overall condition assessment and suggest a PSA grade from 1 to 10. "
        "Explain your reasoning briefly."
    ),
}

def assess_condition(front_path: str | Path, back_path: Optional[str | Path] = None) -> dict:
    """
    Run all condition prompts on front (and optionally back) image.
    Returns dict keyed by dimension name.
    """
    results = {}
    print("Condition Assessment — Front")
    print("═" * 60)
    for dim, prompt in CONDITION_PROMPTS.items():
        ans = paligemma(prompt, front_path, max_tokens=256)
        results[f"front_{dim}"] = ans
        print(f"\n▸ {dim.upper()} (front)")
        for line in textwrap.wrap(ans, width=80):
            print(f"  {line}")

    if back_path:
        back_corner_prompt = (
            "Examine the four corners and all four edges of the BACK of this trading card. "
            "Describe any whitening, fraying, or rounding. "
            "Rate overall back condition: Pristine / Minor Wear / Moderate Wear / Heavy Wear."
        )
        back_surface_prompt = (
            "Examine the back surface and centering of this trading card. "
            "Describe any scratches, print lines, or centering issues on the back."
        )
        print("\nCondition Assessment — Back")
        print("═" * 60)
        for name, prompt in [("back_corners_edges", back_corner_prompt),
                              ("back_surface", back_surface_prompt)]:
            ans = paligemma(prompt, back_path, max_tokens=256)
            results[name] = ans
            print(f"\n▸ {name.upper()}")
            for line in textwrap.wrap(ans, width=80):
                print(f"  {line}")
    return results


condition = assess_condition(TEST_FRONT, TEST_BACK)

## Task 4 — OCR (card text, number, HP)

In [ ]:
def ocr_card(image_path: str | Path) -> str:
    """Run PaliGemma's native OCR task."""
    raw = paligemma("ocr", image_path, max_tokens=512)
    print("OCR output:")
    print("-" * 50)
    print(raw)
    return raw


ocr_result = ocr_card(TEST_FRONT)

## Task 5 — PSA Grade Prediction (Chain-of-Thought)

In [ ]:
GRADE_PROMPT_FRONT = """\
You are a PSA-certified card grading expert evaluating a Pokémon trading card.

Analyze this card front image and score each dimension:

1. Centering (L/R and T/B ratios, max allowed: 55/45 for PSA 10)
2. Corners (all four: sharp vs worn)
3. Edges (all four: clean vs chipped)
4. Surface (scratches, print lines, gloss loss)

Then output:
CENTERING: <score 1-10 with brief rationale>
CORNERS: <score 1-10 with brief rationale>
EDGES: <score 1-10 with brief rationale>
SURFACE: <score 1-10 with brief rationale>
LIMITING FACTOR: <the weakest dimension>
PREDICTED PSA GRADE: <integer 1-10>
CONFIDENCE: <low | medium | high>
"""

GRADE_PROMPT_BACK = """\
You are a PSA-certified card grading expert evaluating the BACK of a Pokémon trading card.

Analyze corners, edges, surface, and centering on this back image.
Then output:
CORNERS: <score 1-10>
EDGES: <score 1-10>
SURFACE: <score 1-10>
CENTERING: <score 1-10>
BACK GRADE CONTRIBUTION: <integer 1-10>
"""

def grade_prediction(
    front_path: str | Path,
    back_path: Optional[str | Path] = None,
) -> dict:
    """Predict PSA grade using chain-of-thought prompts on front (+ optional back)."""
    print("PSA Grade Prediction")
    print("═" * 60)

    front_raw = paligemma(GRADE_PROMPT_FRONT, front_path, max_tokens=512)
    print("\n── FRONT ANALYSIS ──")
    print(front_raw)

    back_raw = None
    if back_path:
        back_raw = paligemma(GRADE_PROMPT_BACK, back_path, max_tokens=256)
        print("\n── BACK ANALYSIS ──")
        print(back_raw)

    # Parse final grade
    grade_match = re.search(r'PREDICTED PSA GRADE[:\s]+([0-9\.]+)', front_raw, re.I)
    conf_match  = re.search(r'CONFIDENCE[:\s]+(low|medium|high)', front_raw, re.I)
    grade = float(grade_match.group(1)) if grade_match else None
    conf  = conf_match.group(1).lower() if conf_match else "unknown"

    print(f"\n{'─'*60}")
    if grade:
        stars = "⭐" * max(0, int(round(grade)))
        print(f"  → Predicted PSA Grade: {grade:.0f}/10  {stars}")
        print(f"  → Confidence: {conf}")
    else:
        print("  → Could not parse grade from response.")

    return {"front_analysis": front_raw, "back_analysis": back_raw,
            "grade": grade, "confidence": conf}


grade_result = grade_prediction(TEST_FRONT, TEST_BACK)

## Task 6 — Visual QA Probe (Free Questions)

In [ ]:
PROBE_QUESTIONS = [
    "Is there any visible print line on the front of the card?",
    "Are the card corners sharp or worn?",
    "Is this card properly centered or does it appear off-center?",
    "Does the card surface appear glossy or is the gloss worn?",
    "Is there any crease or bend visible on this card?",
    "What is the dominant color of the card border?",
    "Is this a holographic or non-holographic card?",
]

print("Visual QA Probes")
print("═" * 60)
probe_results = {}
for q in PROBE_QUESTIONS:
    ans = paligemma(q, TEST_FRONT, max_tokens=128)
    probe_results[q] = ans
    short_q = q[:55] + ("..." if len(q) > 55 else "")
    print(f"\nQ: {short_q}")
    print(f"A: {ans}")

## Task 7 — Batch Analysis Across All Card Images

In [ ]:
from pathlib import Path

CARDS_DIR = Path("cards")
fronts = sorted(CARDS_DIR.glob("*_front.*"))

print(f"Found {len(fronts)} front images")

batch_results = []

fig, axes = plt.subplots(len(fronts), 2, figsize=(10, 5 * len(fronts)))
if len(fronts) == 1:
    axes = [axes]

for i, front_path in enumerate(fronts):
    back_path = Path(str(front_path).replace("_front", "_back"))
    if not back_path.exists():
        back_path = None

    print(f"\n{'━'*60}")
    print(f"Card {i}: {front_path.name}")
    print(f"{'━'*60}")

    # Quick identification
    name_ans = paligemma("What is the name of this Pokémon card?", front_path, max_tokens=64)

    # Quick grade
    grade_raw = paligemma(GRADE_PROMPT_FRONT, front_path, max_tokens=512)
    grade_match = re.search(r'PREDICTED PSA GRADE[:\s]+([0-9\.]+)', grade_raw, re.I)
    grade = float(grade_match.group(1)) if grade_match else None
    conf_match  = re.search(r'CONFIDENCE[:\s]+(low|medium|high)', grade_raw, re.I)
    conf  = conf_match.group(1).lower() if conf_match else "?"

    limiting_match = re.search(r'LIMITING FACTOR[:\s]+([^\n]+)', grade_raw, re.I)
    limiting = limiting_match.group(1).strip() if limiting_match else "?"

    print(f"  Name:          {name_ans}")
    print(f"  Grade:         {grade}/10  (confidence: {conf})")
    print(f"  Limiting:      {limiting}")

    # Draw front with grade overlay
    img_f = Image.open(front_path)
    axes[i][0].imshow(img_f)
    grade_label = f"PSA {grade:.0f}" if grade else "PSA ?"
    axes[i][0].set_title(f"{name_ans}\n{grade_label} | conf: {conf}", fontsize=9)
    axes[i][0].axis("off")

    if back_path:
        img_b = Image.open(back_path)
        axes[i][1].imshow(img_b)
        axes[i][1].set_title("Back", fontsize=9)
        axes[i][1].axis("off")
    else:
        axes[i][1].axis("off")
        axes[i][1].text(0.5, 0.5, "No back image", ha="center", va="center", fontsize=10)

    batch_results.append({
        "path": str(front_path),
        "name": name_ans,
        "grade": grade,
        "confidence": conf,
        "limiting_factor": limiting,
        "raw_analysis": grade_raw,
    })

plt.suptitle("PaliGemma Batch Card Analysis", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Task 8 — Condition Comparison: Front vs Back Side-by-Side

In [ ]:
def full_card_report(
    front_path: str | Path,
    back_path: Optional[str | Path] = None,
) -> None:
    """Print a nicely formatted grading report for one card."""
    front_path = Path(front_path)

    # Identification
    card_name = paligemma("What is the name of this Pokémon card?", front_path, max_tokens=64)
    card_set  = paligemma("What Pokémon TCG set does this card belong to?", front_path, max_tokens=64)
    card_num  = paligemma("What is the card number and total?", front_path, max_tokens=32)

    # Grade prediction
    grade_raw = paligemma(GRADE_PROMPT_FRONT, front_path, max_tokens=512)
    grade_m   = re.search(r'PREDICTED PSA GRADE[:\s]+([0-9\.]+)', grade_raw, re.I)
    conf_m    = re.search(r'CONFIDENCE[:\s]+(low|medium|high)', grade_raw, re.I)
    lim_m     = re.search(r'LIMITING FACTOR[:\s]+([^\n]+)', grade_raw, re.I)

    grade    = float(grade_m.group(1)) if grade_m else None
    conf     = conf_m.group(1) if conf_m else "?"
    limiting = lim_m.group(1).strip() if lim_m else "?"

    # Visual display
    n_cols = 2 if back_path else 1
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 7))
    if n_cols == 1:
        axes = [axes]

    axes[0].imshow(Image.open(front_path))
    axes[0].set_title("Front", fontsize=9)
    axes[0].axis("off")

    if back_path:
        axes[1].imshow(Image.open(back_path))
        axes[1].set_title("Back", fontsize=9)
        axes[1].axis("off")

    grade_str = f"PSA {grade:.0f}" if grade else "PSA ?"
    fig.suptitle(f"{card_name} — {card_set} {card_num}\n{grade_str} | Confidence: {conf}",
                 fontsize=11)
    plt.tight_layout()
    plt.show()

    # Text report
    report = f"""
╔══════════════════════════════════════════════════════════╗
║  PALIGEMMA GRADING REPORT
╠══════════════════════════════════════════════════════════╣
║  Card:           {card_name[:40]}
║  Set:            {card_set[:40]}
║  Number:         {card_num[:20]}
║  Predicted PSA:  {grade_str}
║  Confidence:     {conf}
║  Limiting:       {limiting[:40]}
╚══════════════════════════════════════════════════════════╝

── RAW ANALYSIS ──
{grade_raw}
"""
    print(report)


# Run on the test card
full_card_report(TEST_FRONT, TEST_BACK)

## Task 9 — Prompt Engineering Comparison

Compare different prompt strategies for grade prediction on the same card.

In [ ]:
PROMPT_VARIANTS = {
    "simple": "What PSA grade would you give this card? Answer with just a number from 1-10.",
    "chain_of_thought": GRADE_PROMPT_FRONT,
    "expert_persona": (
        "You are a veteran PSA grader with 20 years of experience. "
        "Grade this card from 1-10 based on corners, edges, surface, and centering. "
        "Be precise and strict. Provide the grade as 'GRADE: X' at the end."
    ),
    "condition_first": (
        "First describe every visible defect on this trading card in detail. "
        "Then based on those defects, assign a PSA grade from 1-10. "
        "Format: DEFECTS: ... | GRADE: X"
    ),
}

print("Prompt Variant Comparison")
print("═" * 70)

variant_results = {}
for name, prompt in PROMPT_VARIANTS.items():
    ans = paligemma(prompt, TEST_FRONT, max_tokens=512)
    variant_results[name] = ans

    # Try to extract a grade
    g = re.search(r'(?:GRADE[:\s]+|grade[:\s]+|PSA[:\s]+)([0-9\.]+)', ans)
    grade_found = float(g.group(1)) if g else None

    print(f"\n▸ Variant: '{name}'")
    print(f"  Extracted grade: {grade_found}")
    print(f"  Response preview: {ans[:200]}..." if len(ans) > 200 else f"  Response: {ans}")

# Summary table
print("\n" + "─"*40)
print("SUMMARY")
print("─"*40)
for name, ans in variant_results.items():
    g = re.search(r'(?:GRADE[:\s]+|PSA[:\s]+|PREDICTED PSA GRADE[:\s]+)([0-9\.]+)', ans, re.I)
    grade_found = float(g.group(1)) if g else "N/A"
    print(f"  {name:<20} → PSA {grade_found}")

## Task 10 — Interactive Single-Image Analysis

Configure `ANALYZE_IMAGE` below to test any card image.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
ANALYZE_IMAGE = "cards/image1_front.jpeg"   # ← change to any card image
ANALYZE_BACK  = "cards/image1_back.jpeg"    # ← set to None if no back
CUSTOM_QUESTION = "Does this card have any print lines on the surface?"  # ← custom VQA
# ─────────────────────────────────────────────────────────────────────────────

print(f"Analyzing: {ANALYZE_IMAGE}")
print("─" * 50)

# 1. Caption
cap = paligemma("caption en", ANALYZE_IMAGE)
print(f"Caption:     {cap}")

# 2. Grade
grade_raw = paligemma(GRADE_PROMPT_FRONT, ANALYZE_IMAGE, max_tokens=512)
grade_m = re.search(r'PREDICTED PSA GRADE[:\s]+([0-9\.]+)', grade_raw, re.I)
grade = float(grade_m.group(1)) if grade_m else None
print(f"Grade:       PSA {grade}")

# 3. Custom question
custom_ans = paligemma(CUSTOM_QUESTION, ANALYZE_IMAGE)
print(f"Custom Q:    {CUSTOM_QUESTION}")
print(f"Answer:      {custom_ans}")

# 4. Full report
back_arg = ANALYZE_BACK if ANALYZE_BACK and Path(ANALYZE_BACK).exists() else None
full_card_report(ANALYZE_IMAGE, back_arg)